# LLM Tone & Safety Alignment with DPO (Single L4 GPU Runner)

Trains **`HuggingFaceH4/zephyr-7b-sft-full`** (an SFT-only 7B model) with **4-bit QLoRA DPO** on a balanced 50/50 mix of `PKU-SafeRLHF` + `UltraFeedback`.

- Set `SANITY_CHECK = True` first (~15–20 min, ~0.6 CCU on 1× L4) to verify `eval_loss` drops well below `0.693` and preference accuracy exceeds `70%`.
- Then set `SANITY_CHECK = False` for the full 8,000-sample run (~1.5–2 hrs, ~3.5 CCUs on 1× L4).


In [ ]:
# 1. Verify Single GPU & Clone/Sync Repo
import os, torch

assert torch.cuda.is_available(), 'No GPU detected! Switch Colab runtime to NVIDIA L4.'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}')

REPO_URL = 'https://github.com/manohar135/LLM-Tone-Alignment-DPO.git'
REPO_DIR = 'LLM-Tone-Alignment-DPO'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull


In [ ]:
# 2. Install Dependencies
!pip install -q -U -r requirements.txt


In [ ]:
# 3. Prepare Balanced Preference Dataset (50% PKU-SafeRLHF + 50% UltraFeedback)
SANITY_CHECK = True  # Switch to False for full 8,000-sample run
FLAG = '--sanity' if SANITY_CHECK else ''

!python data_preparation.py {FLAG}


In [ ]:
# 4. Train DPO with 4-bit QLoRA & Plot Training Curves
!python train_dpo.py {FLAG}

from IPython.display import Image, display
if os.path.exists('outputs/training_curves.png'):
    display(Image('outputs/training_curves.png'))


In [ ]:
# 5. Evaluate Base (SFT) vs. DPO-Aligned on 60 Curated + 40 Adversarial PKU Prompts
!python evaluate.py --batch-size 8 --hf-safety-n 40

if os.path.exists('outputs/eval_comparison.png'):
    display(Image('outputs/eval_comparison.png'))


In [ ]:
# 6. Zip & Download Trained Adapter, Metrics, and Charts
!zip -r tone_align_outputs.zip outputs/dpo_adapter outputs/training_metrics.json outputs/eval_results.json outputs/training_curves.png outputs/eval_comparison.png -x 'outputs/dpo_adapter/checkpoint-*/*'

try:
    from google.colab import files
    files.download('tone_align_outputs.zip')
except ImportError:
    print('Saved tone_align_outputs.zip locally.')


In [ ]:
# 7. (Optional) Launch Interactive Gradio Side-by-Side Demo
# !python app.py --share
